In [1]:
import sys
import os
# Tell Python to look two folders up (the project root) so it can find 'src'
sys.path.append(os.path.abspath('../../'))

import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from src.utils.paths import PROCESSED_DATA_DIR

# Load the clean, scaled data your preprocessing script created
X_train = pd.read_csv(PROCESSED_DATA_DIR / "X_train.csv")
X_test = pd.read_csv(PROCESSED_DATA_DIR / "X_test.csv")
y_train = pd.read_csv(PROCESSED_DATA_DIR / "y_train.csv").values.ravel()
y_test = pd.read_csv(PROCESSED_DATA_DIR / "y_test.csv").values.ravel()

In [2]:
# Initialize the KNN classifier with k=3
knn_estimator = KNeighborsClassifier(n_neighbors=3)

# Train the model on the training data
knn_estimator.fit(X_train, y_train)

print("KNN Model trained successfully!")

KNN Model trained successfully!


In [3]:
# Use the trained model to predict the test data
y_pred = knn_estimator.predict(X_test)
y_prob = knn_estimator.predict_proba(X_test)[:, 1] # Probabilities for ROC AUC

# Evaluate the results
print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC AUC Score:", roc_auc_score(y_test, y_prob))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.965275278877294
ROC AUC Score: 0.9569977492213215

Classification Report:
               precision    recall  f1-score   support

           0       0.97      0.98      0.98      4115
           1       0.94      0.92      0.93      1443

    accuracy                           0.97      5558
   macro avg       0.96      0.95      0.95      5558
weighted avg       0.97      0.97      0.97      5558



In [7]:
from sklearn.model_selection import RandomizedSearchCV, cross_val_score, RepeatedStratifiedKFold
from sklearn.neighbors import KNeighborsClassifier

# 1. Define the parameter grid to search through
param_grid = {
    'n_neighbors': [3, 5, 7, 9, 11],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

In [10]:

# 2. Set up the Inner and Outer loops
# We use StratifiedKFold because your classes are imbalanced (more Introverts than Extroverts)
cv_inner = RepeatedStratifiedKFold(n_splits=3, n_repeats=3, random_state=42)
cv_outer = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)

# 3. Define the base model
knn = KNeighborsClassifier()

# 4. INNER LOOP: RandomizedSearchCV
search = RandomizedSearchCV(
    knn,
    param_distributions=param_grid,
    n_iter=15,
    scoring='accuracy',
    cv=cv_inner,
    random_state=42,
    n_jobs=-1
)

# 5. OUTER LOOP: cross_val_score evaluates the true performance
# Note: We use X_train and y_train here. The outer loop will split this further into Train/Test folds!
nested_scores = cross_val_score(search, X_train, y_train, scoring='accuracy', cv=cv_outer, n_jobs=-1)

In [11]:
print("Nested CV Accuracy Scores for each fold:", nested_scores)
print("True Expected Accuracy:", nested_scores.mean())

# 6. Finally, train the best model on the FULL training set to see what parameters won
search.fit(X_train, y_train)
print("\nBest Hyperparameters Found:", search.best_params_)

Nested CV Accuracy Scores for each fold: [0.97494217 0.96953336 0.96529117 0.96799074 0.96760509 0.97224364
 0.96683378 0.96799074 0.97030467 0.96799074 0.96838859 0.96760509
 0.97261859 0.9683764  0.9683764 ]
True Expected Accuracy: 0.9690727452268295

Best Hyperparameters Found: {'weights': 'uniform', 'n_neighbors': 11, 'metric': 'manhattan'}
